# Practice 40 — pandas + NumPy

In [1]:
import pandas as pd
import numpy as np
import seaborn as sns

---
## Level 1 — vectorized feature building

Load `sns.load_dataset('diamonds')`. No `.apply()`.

1. Add `volume_approx` = `x * y * z` and `price_per_volume` = `price / volume_approx`. Some rows have 0 in x, y, or z — replace those with `np.nan` using `np.where` after computing.
2. Use `np.select()` to add `shape_class`: `'cube-like'` (all of x, y, z within 10% of each other — hint: use `x/y` and `x/z` ratios), `'flat'` (z < 2), `'elongated'` otherwise.
3. Add `log_price` and `log_carat`. What is `np.corrcoef` between them? Is it stronger than raw `price` vs `carat`?
4. Convert `cut`, `color`, `clarity` to `category`. Print memory before and after in MB. Use a list comprehension to collect the names of all columns that are now `category` dtype.

In [29]:
# Your code here
diamonds = sns.load_dataset('diamonds')
diamonds['volumn_approx'] = diamonds['x'] * diamonds['y'] * diamonds['z']

diamonds['price_per_volume'] = diamonds['price']/diamonds['volumn_approx']
diamonds['x'] = np.where(diamonds['x']==0, np.nan, diamonds['x'])
diamonds['y'] = np.where(diamonds['y']==0, np.nan, diamonds['y'])
diamonds['z'] = np.where(diamonds['z']==0, np.nan, diamonds['z'])

xry = diamonds['x']/diamonds['y']
xrz = diamonds['x']/diamonds['z']
conds = [(xry>0.9) & (xry<1.1) & (xrz<1.1) & (xrz>0.9), diamonds['z']<2]
chos = ['cube_like','flat']
diamonds['shape_class'] = np.select(conds, chos, default='elongated')

diamonds['log_price'] =np.log(diamonds['price'])
diamonds['log_carat'] = np.log(diamonds['carat'])
logc = np.corrcoef(diamonds['log_price'], diamonds['log_carat'])[0,1]
c = np.corrcoef(diamonds['price'], diamonds['carat'])[0,1]

print(f'log cor: {logc:.2f}')
print(f'raw cor: {c:.2f}')


mb = diamonds.memory_usage(deep=True).sum()/1024/1024
diamonds[['cut','color','clarity']] = diamonds[['cut','color','clarity']].astype('category')
ma = diamonds.memory_usage(deep=True).sum()/1024/1024
print(f'before: {mb:2f}mb')
print(f'after: {ma:2f}mb')

[name for name in diamonds.columns if diamonds[name].dtypes == 'category'] 


log cor: 0.97
raw cor: 0.92
before: 8.078256mb
after: 8.078256mb


['cut', 'color', 'clarity']

---
## Level 2 — rewrite + comprehensions

The code below uses `.apply()` and plain loops. Rewrite everything — no `.apply()`, no loops.

```python
mpg = sns.load_dataset('mpg').dropna()

# A — vectorize this
mpg['kpl'] = mpg['mpg'].apply(lambda x: x * 0.425144)

# B — vectorize this
mpg['decade'] = mpg['model_year'].apply(
    lambda x: '70s' if x < 80 else '80s')

# C — vectorize this
mpg['heavy'] = mpg.apply(
    lambda row: True if row['weight'] > row['weight'].mean() else False, axis=1)

# D — rewrite without a loop
origins = []
for o in mpg['origin']:
    if o != 'usa':
        origins.append(o)

# E — rewrite without a loop
result = {}
for name, val in zip(mpg['name'], mpg['mpg']):
    if val > 35:
        result[name] = val
```

After rewriting D and E: what type does each produce — list, set, or dict? When would you choose each?

In [ ]:
mpg = sns.load_dataset('mpg').dropna()

# Your rewrites here
mpg['kpl'] = mpg['mpg']*0.425144

mpg['decade'] = np.where(mpg['model_year']<80, '70s','80s')

mpg['heavy'] = mpg['weight']>mpg['weight'].mean()

origins = [o for o in mpg['origin'] if o != 'usa']

result = {name: val for name, val in zip(mpg['name'], mpg['mpg']) if val >35}

# D is list E is dict


---
## Level 3 — pipeline

Load `sns.load_dataset('titanic')`. Write a `.pipe()` chain — no `.apply()`.

- **`features(df)`** — add `age_group` with `np.select()`: `'child'` (age < 18), `'adult'` (18–59), `'senior'` (60+), default `'unknown'`. Add `fare_per_person` = `fare / (sibsp + parch + 1)`. Convert `sex`, `pclass`, and `age_group` to `category`.
- **`score(df)`** — add `risk_score` vectorized: start from 0, then add 1 if `pclass == 3`, add 1 if `age_group == 'senior'`, add 1 if `fare_per_person < 10`. (Three separate vectorized additions — no `np.select` needed.)

After the chain:
- What fraction of passengers with `risk_score == 3` survived? Use `.loc` and `np.mean()`.
- Pivot table: mean `survived` by `sex` × `age_group`, `observed=True`. Stack it and find the `(sex, age_group)` with the highest survival rate.
- Use a dict comprehension to build `{age_group: mean_fare_per_person}` — iterate over the groupby result.

In [47]:
# Your code here
titanic = sns.load_dataset('titanic')

def features(df):
    conds = [df['age']<18, df['age']<59, df['age']<np.inf]
    chois = ['child','adult','senior']
    df['age_group'] = np.select(conds, chois, default='unknown')
    df['fare_per_person'] = df['fare']/(df['sibsp']+df['parch']+1)
    df[['sex','pclass','age_group']] = df[['sex','pclass','age_group']].astype("category")
    return df 


def score(df):
    df['risk_score'] = 0
    df.loc[df['pclass']==3,'risk_score'] += 1
    df.loc[df['age_group'] == 'senior','risk_score'] += 1
    df.loc[df['fare_per_person']<10, 'risk_score'] += 1
    return df 


res = titanic.copy().pipe(features).pipe(score)

np.mean(res.loc[res['risk_score']==3,'survived'])

p = res.pivot_table(
    index = 'sex',
    columns = 'age_group',
    values = 'survived',
    observed = True
)
p.stack().idxmax()

m = res.groupby('age_group', observed=True)['fare_per_person'].mean()

{a:m for a, m in zip(m.index, m)}

{'adult': 22.653242934014795,
 'child': 11.024569521702487,
 'senior': 24.929391071428572,
 'unknown': 15.940015353535355}